# AILS Quick Start Guide

This notebook provides a quick introduction to the Adaptive Incremental Line Search (AILS) algorithm.

**Author:** Amr Elshahed  
**Institution:** Universiti Sains Malaysia

---

## 1. Setup and Installation

First, let's install the required dependencies.

In [ ]:
# Install required packages (run once)
# !pip install numpy scipy pandas matplotlib seaborn tqdm

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

# Import AILS core module
from ails_core import (
    AILSPathfinder, AILSConfig, GridGenerator,
    run_benchmark, compute_statistics, compute_improvement
)

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("All modules loaded successfully!")

## 2. Create a Simple Grid

Let's create a simple grid environment and visualize it.

In [ ]:
# Create a 50x50 grid with 20% obstacles
grid = GridGenerator.generate_random(size=50, density=0.2, seed=42)

print(f"Grid size: {grid.shape}")
print(f"Total cells: {grid.size}")
print(f"Traversable cells: {np.sum(grid == 0)}")
print(f"Obstacle cells: {np.sum(grid == 1)}")
print(f"Actual obstacle density: {np.mean(grid):.2%}")

In [ ]:
# Visualize the grid
plt.figure(figsize=(8, 8))
plt.imshow(grid, cmap='binary', origin='upper')
plt.colorbar(label='Obstacle (1) / Free (0)')
plt.title('Grid Environment (50x50, 20% obstacles)')
plt.xlabel('Column')
plt.ylabel('Row')
plt.tight_layout()
plt.show()

## 3. Basic Pathfinding

Let's find a path using different algorithms and compare them.

In [ ]:
# Initialize the pathfinder
pathfinder = AILSPathfinder(grid)

# Find start and goal positions (random traversable cells)
traversable = np.argwhere(grid == 0)
np.random.seed(123)
idx = np.random.choice(len(traversable), 2, replace=False)
start = tuple(traversable[idx[0]])
goal = tuple(traversable[idx[1]])

print(f"Start position: {start}")
print(f"Goal position: {goal}")

In [ ]:
# Compare all methods
results = pathfinder.compare_methods(start, goal)

# Display results
print("\n" + "="*70)
print("PATHFINDING RESULTS")
print("="*70)
print(f"{'Method':<20} {'Time (ms)':<12} {'Nodes':<10} {'Cost':<10} {'Found'}")
print("-"*70)

for method, result in results.items():
    status = 'Yes' if result.path_found else 'No'
    print(f"{method:<20} {result.time_ms:<12.3f} {result.nodes_visited:<10} {result.cost:<10.2f} {status}")

In [ ]:
# Visualize the paths
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

colors = ['red', 'blue', 'green', 'orange', 'purple', 'cyan']

for ax, (method, result), color in zip(axes, results.items(), colors):
    ax.imshow(grid, cmap='Greys', origin='upper')
    
    if result.path_found and result.path:
        path = np.array(result.path)
        ax.plot(path[:, 1], path[:, 0], color=color, linewidth=2, alpha=0.8)
    
    ax.scatter(start[1], start[0], c='green', s=100, marker='s', label='Start', zorder=5)
    ax.scatter(goal[1], goal[0], c='red', s=100, marker='*', label='Goal', zorder=5)
    
    ax.set_title(f"{method}\nNodes: {result.nodes_visited}, Time: {result.time_ms:.2f}ms")
    ax.legend(loc='upper right')

plt.tight_layout()
plt.suptitle('Path Comparison Across Different Algorithms', y=1.02, fontsize=14)
plt.show()

## 4. AILS Configuration

AILS has several configurable parameters that affect its performance.

In [ ]:
# Display default configuration
default_config = AILSConfig()

print("AILS Default Configuration:")
print("="*40)
print(f"r_min (minimum radius): {default_config.r_min}")
print(f"r_max (maximum radius): {default_config.r_max}")
print(f"alpha (density scaling): {default_config.alpha}")
print(f"window_size (density window): {default_config.window_size}")
print(f"max_iterations: {default_config.max_iterations}")

In [ ]:
# Test with custom configuration
custom_config = AILSConfig(
    r_min=3,        # Smaller minimum radius
    r_max=20,       # Larger maximum radius
    alpha=0.6,      # Different scaling
    window_size=5,  # Smaller window
    max_iterations=3
)

# Create pathfinder with custom config
pathfinder_custom = AILSPathfinder(grid, config=custom_config)

# Compare default vs custom
result_default = pathfinder.find_path_ails(start, goal, strategy='adaptive')
result_custom = pathfinder_custom.find_path_ails(start, goal, strategy='adaptive')

print("\nDefault Config vs Custom Config:")
print("="*50)
print(f"{'Config':<15} {'Time (ms)':<12} {'Nodes':<10} {'Corridor Size'}")
print("-"*50)
print(f"{'Default':<15} {result_default.time_ms:<12.3f} {result_default.nodes_visited:<10} {result_default.corridor_size}")
print(f"{'Custom':<15} {result_custom.time_ms:<12.3f} {result_custom.nodes_visited:<10} {result_custom.corridor_size}")

## 5. Different Grid Types

AILS can work with various obstacle patterns. Let's explore different grid types.

In [ ]:
# Generate different types of grids
grid_types = {
    'Random': GridGenerator.generate_random(100, density=0.25, seed=42),
    'Clustered': GridGenerator.generate_clustered(100, density=0.25, num_clusters=15, seed=42),
    'Maze': GridGenerator.generate_maze(101, seed=42),
    'Room': GridGenerator.generate_room(100, num_rooms=6, seed=42),
    'Open': GridGenerator.generate_open(100, density=0.1, seed=42)
}

# Visualize all grid types
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, (name, g) in zip(axes, grid_types.items()):
    ax.imshow(g, cmap='binary', origin='upper')
    density = np.mean(g)
    ax.set_title(f"{name}\n(Density: {density:.1%})")
    ax.axis('off')

plt.tight_layout()
plt.suptitle('Different Grid Types', y=1.02, fontsize=14)
plt.show()

In [ ]:
# Test AILS on different grid types
print("\nAILS Performance on Different Grid Types:")
print("="*70)
print(f"{'Grid Type':<15} {'A* Nodes':<12} {'AILS Nodes':<12} {'Reduction':<12} {'Time Saved'}")
print("-"*70)

for name, g in grid_types.items():
    # Find valid start and goal
    traversable = np.argwhere(g == 0)
    if len(traversable) < 2:
        print(f"{name:<15} No valid paths available")
        continue
    
    np.random.seed(42)
    idx = np.random.choice(len(traversable), 2, replace=False)
    start = tuple(traversable[idx[0]])
    goal = tuple(traversable[idx[1]])
    
    pf = AILSPathfinder(g)
    astar_result = pf.find_path_standard(start, goal, 'astar')
    ails_result = pf.find_path_ails(start, goal, strategy='adaptive')
    
    if astar_result.path_found and ails_result.path_found:
        node_reduction = (1 - ails_result.nodes_visited / astar_result.nodes_visited) * 100
        time_saved = (1 - ails_result.time_ms / astar_result.time_ms) * 100
        print(f"{name:<15} {astar_result.nodes_visited:<12} {ails_result.nodes_visited:<12} "
              f"{node_reduction:<12.1f}% {time_saved:.1f}%")
    else:
        print(f"{name:<15} Path not found")

## 6. Quick Benchmark

Let's run a quick benchmark to see how AILS compares to standard algorithms.

In [ ]:
# Run a quick benchmark
grid = GridGenerator.generate_random(100, density=0.25, seed=42)

print("Running benchmark with 50 random start-goal pairs...")
print("This may take a few seconds...\n")

results = run_benchmark(grid, num_pairs=50, seed=42,
                       algorithms=['A*', 'AILS-Base', 'AILS-Adaptive'])

stats = compute_statistics(results)

# Display statistics
print("\nBenchmark Results:")
print("="*80)
print(f"{'Method':<20} {'Mean Time':<12} {'Std Time':<10} {'Mean Nodes':<12} {'Success Rate'}")
print("-"*80)

for method, s in stats.items():
    print(f"{method:<20} {s['time_mean']:<12.3f} {s['time_std']:<10.3f} "
          f"{s['nodes_mean']:<12.1f} {s['success_rate']:.1f}%")

In [ ]:
# Compute improvements over A*
improvements = compute_improvement(stats, baseline='A*')

print("\nImprovement over A*:")
print("="*50)

for method, imp in improvements.items():
    print(f"\n{method}:")
    print(f"  Node reduction: {imp['nodes_improvement']:.1f}%")
    print(f"  Time improvement: {imp['time_improvement']:.1f}%")

## 7. Summary

This quick start notebook covered:

1. **Setup**: How to import and configure AILS
2. **Basic Usage**: Creating grids and finding paths
3. **Configuration**: AILS parameters and their effects
4. **Grid Types**: Different obstacle patterns AILS can handle
5. **Benchmarking**: Comparing AILS with standard algorithms

### Next Steps:

- **02_experiments.ipynb**: Run comprehensive experiments
- **03_statistical_analysis.ipynb**: Detailed statistical analysis
- **04_visualization.ipynb**: Create publication-quality figures

---

**Key Findings:**
- AILS reduces the search space significantly by focusing on a corridor
- The adaptive strategy adjusts corridor width based on local obstacle density
- Performance gains are especially notable in larger grids with moderate obstacle density